# Housing Price Prediction — Model Development

## 1. Project Overview

The objective of this project is to predict residential property selling prices using historical housing transaction data from the SSU Data Dash competition.

Two prediction tracks are considered:

- **Off-market:** Predict selling prices without using listing prices.
- **List-aware:** Predict selling prices with listing prices available.

This notebook focuses on developing and evaluating machine learning models for the **Off-market** track.

The modeling workflow includes data preparation, feature engineering, model training, time-based validation, and performance evaluation using Median Absolute Percentage Error (MdAPE).

In [3]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

project_root = Path.cwd()
if not (project_root / "data.py").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

import data
import features

## 2. Data Loading

The dataset is loaded using the competition's provided `data.load()` function.

It contains **150,000 observations and 22 variables**, including property characteristics, geographic information, listing details, and selling prices.

The dataset is used to train and validate housing price prediction models.

In [4]:
listings = data.load("listings")

print("Dataset shape:", listings.shape)
display(listings.head())

/Users/hyewonerinyoon/Desktop/My Stuff/Projects/SSU Data Dash/SSU-Data-Dash/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset shape: (150000, 22)


,ml_num,sold_price,list_price,date_listed,date_sold,days_on_market,house_type_name,house_style,bedrooms,washrooms,...,house_area,lot_front,lot_depth,municipality,community,latitude,longitude,external_avm_estimate,external_avm_estimate_date,remarks
0,SYN-0023247,364100.0,371200.0,2020-12-29,2021-01-01,3,Condo Apt,Apartment,1,1,...,622,NaN,NaN,Toronto,Sumacpark,43.602039,-79.261184,366200.0,2021-02-14,Looking For More Space In A Turnkey Condo? Thi...
1,SYN-0156988,599000.0,595800.0,2020-12-26,2021-01-01,6,Condo Apt,Apartment,2,1,...,649,NaN,NaN,Toronto,Hickorymills,43.695388,-79.510865,622800.0,2021-12-26,"Imagine a quiet evening on Sequoia Trail, wher..."
2,SYN-0087706,492900.0,462200.0,2020-12-22,2021-01-01,10,Condo Apt,Apartment,2,1,...,900-1099,NaN,NaN,North York,Hazelmills,43.740512,-79.386216,NaN,2020-12-03,"Discover this inviting 2-bedroom, 1-bath condo..."
3,SYN-0128154,532300.0,524900.0,2020-12-19,2021-01-01,13,Condo Apt,Apartment,1,1,...,876,NaN,NaN,Scarborough,Sumacbank,43.643774,-79.470180,573600.0,2021-02-22,Sumacbank presents this 1-bedroom condo on Mag...
4,SYN-0118596,564100.0,558300.0,2020-12-05,2021-01-01,27,Condo Apt,Apartment,1,1,...,514,NaN,NaN,Toronto,Ashworth,43.696172,-79.274137,513000.0,2021-02-05,"Morning coffee at the kitchen island, then a s..."


## 3. Time-Based Train–Validation Split

To prevent future information from being used during training, the dataset is split chronologically based on `date_sold`.

For the initial experiment:

- **Training set:** 147,709 homes sold before May 1, 2026
- **Validation set:** 2,291 homes sold during May 2026

Unlike a random train–test split, this approach better simulates the competition's prediction task, where models are trained on historical transactions and evaluated on future sales.

In [5]:
validation_month = pd.Timestamp("2026-05-01")
next_month = pd.Timestamp("2026-06-01")

train = listings[
    listings["date_sold"] < validation_month
].copy()

valid = listings[
    (listings["date_sold"] >= validation_month) &
    (listings["date_sold"] < next_month)
].copy()

print("Training:", train.shape)
print("Validation:", valid.shape)

Training: (147709, 22)
Validation: (2291, 22)


## 4. Feature Preparation

The competition's `off_market_features()` function is used to transform property information into numerical features.

The initial model uses **13 features**, including:

- Property size, bedrooms, washrooms, and parking
- Geographic coordinates (latitude and longitude)
- Time-related information derived from listing dates
- One-hot encoded property types

The target variable, `sold_price`, is log-transformed to account for the right-skewed distribution of housing prices.

The `hide_answers()` function is applied to prevent unavailable information from entering the model.

In [6]:
X_train = features.off_market_features(
    data.hide_answers(train, "off_market")
)

X_valid = features.off_market_features(
    data.hide_answers(valid, "off_market")
)

y_train = np.log(train["sold_price"])

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)

display(X_train.head())

X_train: (147709, 13)
X_valid: (2291, 13)
y_train: (147709,)


,square_feet,bedrooms,washrooms,parking,latitude,longitude,months,is_Condo Apt,is_Detached,is_Semi-Detached,is_Condo Townhouse,is_Freehold Townhouse,is_Co-Op Apt
0,622.0,1,1,0,43.602039,-79.261184,-0.098555,1,0,0,0,0,0
1,649.0,2,1,0,43.695388,-79.510865,-0.197109,1,0,0,0,0,0
2,999.5,2,1,1,43.740512,-79.386216,-0.328515,1,0,0,0,0,0
3,876.0,1,1,1,43.643774,-79.470180,-0.427070,1,0,0,0,0,0
4,514.0,1,1,1,43.696172,-79.274137,-0.886991,1,0,0,0,0,0


## 5. Initial Model — Histogram Gradient Boosting

A `HistGradientBoostingRegressor` is trained using the 13 starter features.

Gradient Boosting combines multiple decision trees sequentially, with each new tree helping to improve the model's predictions.

The model is trained on log-transformed selling prices using the default hyperparameters and `random_state=42` for reproducibility.

This initial model serves as the benchmark for subsequent feature engineering and hyperparameter tuning experiments.

In [8]:
model = HistGradientBoostingRegressor(
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


## 6. Model Prediction and Evaluation

The trained model predicts log selling prices for the validation set. Predictions are then converted back to dollar values using the exponential function.

Model performance is evaluated using Median Absolute Percentage Error (MdAPE):

**MdAPE = median(|Predicted Price − Actual Price| / Actual Price) × 100**

MdAPE measures the median percentage difference between predicted and actual selling prices. Lower values indicate better predictive performance.

**Initial validation result (May 2026): 15.07% MdAPE.**

In [9]:
predicted_log = model.predict(X_valid)

predicted_price = np.exp(predicted_log)

results = pd.DataFrame({
    "actual": valid["sold_price"].to_numpy(),
    "predicted": predicted_price
})

results["APE"] = (
    abs(results["predicted"] - results["actual"])
    / results["actual"]
)

mdape = results["APE"].median() * 100

print(f"May 2026 MdAPE: {mdape:.2f}%")

display(results.head())

May 2026 MdAPE: 15.07%


,actual,predicted,APE
0,1079700.0,1.332877e+06,0.234488
1,571000.0,5.238113e+05,0.082642
2,390600.0,5.129380e+05,0.313205
3,2182900.0,1.629451e+06,0.253538
4,1352500.0,1.324164e+06,0.020951


## 7. Walk-Forward Validation

To evaluate the model more reliably, walk-forward validation is performed from December 2025 to May 2026.

For each validation month, the model is retrained using only transactions that occurred before that month and evaluated on homes sold during the month.

This method allows model performance to be assessed across multiple time periods rather than relying on a single validation month.

In [10]:
validation_months = pd.period_range(
    "2025-12", "2026-05", freq="M"
)

all_results = []

for month in validation_months:
    start = month.start_time
    end = (month + 1).start_time

    train_month = listings[
        listings["date_sold"] < start
    ]

    valid_month = listings[
        (listings["date_sold"] >= start) &
        (listings["date_sold"] < end)
    ]

    X_train_month = features.off_market_features(
        data.hide_answers(train_month, "off_market")
    )

    X_valid_month = features.off_market_features(
        data.hide_answers(valid_month, "off_market")
    )

    y_train_month = np.log(train_month["sold_price"])

    monthly_model = HistGradientBoostingRegressor(
        random_state=42
    )

    monthly_model.fit(X_train_month, y_train_month)

    predictions = np.exp(
        monthly_model.predict(X_valid_month)
    )

    monthly_results = pd.DataFrame({
        "month": str(month),
        "actual": valid_month["sold_price"].to_numpy(),
        "predicted": predictions
    })

    monthly_results["APE"] = (
        abs(monthly_results["predicted"] - monthly_results["actual"])
        / monthly_results["actual"]
    )

    all_results.append(monthly_results)

    print(
        f"{month}: "
        f"MdAPE = {monthly_results['APE'].median() * 100:.2f}%"
    )

validation_results = pd.concat(
    all_results, ignore_index=True
)

overall_mdape = validation_results["APE"].median() * 100

print(f"\nOverall MdAPE: {overall_mdape:.2f}%")

2025-12: MdAPE = 15.74%
2026-01: MdAPE = 16.07%
2026-02: MdAPE = 14.17%
2026-03: MdAPE = 14.76%
2026-04: MdAPE = 14.43%
2026-05: MdAPE = 15.07%

Overall MdAPE: 14.87%


### Walk-Forward Validation Results

| Validation Month | MdAPE |
|---|---:|
| December 2025 | 15.74% |
| January 2026 | 16.07% |
| February 2026 | 14.17% |
| March 2026 | 14.76% |
| April 2026 | 14.43% |
| May 2026 | 15.07% |
| **Overall** | **14.87%** |

The model achieved an overall MdAPE of **14.87%**, compared with the competition's starter Off-market baseline of **20.84%** over the same six-month validation period.

This represents an improvement of **5.97 percentage points**, equivalent to approximately a **28.6% reduction in MdAPE**.

Performance varies moderately across months, with January 2026 showing the highest error and February 2026 showing the lowest error. Further analysis is needed to determine which property characteristics or market conditions contribute to these differences.

## 8. Experiment 2 — Feature Engineering

### Objective

This experiment investigates whether adding property characteristics and geographic information improves the predictive performance of the initial Gradient Boosting model.

### Motivation

The exploratory data analysis identified several potentially useful relationships:

- Selling prices vary substantially across communities.
- House area is strongly positively correlated with selling price (r = 0.773).
- Lot measurements are available for freehold property types but missing for condo-related properties.
- Property characteristics may explain additional variation in selling prices.

### Proposed Additional Features

- `garage`: Garage information
- `lot_area`: Lot frontage multiplied by lot depth
- `has_lot_info`: Indicator for available lot measurements
- `municipality`: Municipality-level location
- `community`: Neighbourhood-level location
- `house_style`: Property style

### Evaluation Strategy

The improved model will be evaluated using the same December 2025–May 2026 walk-forward validation framework.

Performance will be compared against the initial model's MdAPE of 14.87% to assess whether the additional features and model adjustments improve prediction accuracy.


In [11]:
model = HistGradientBoostingRegressor(
    random_state=42
)

In [12]:
def improved_features(homes):
    # Start with the original 13 features
    X = features.off_market_features(homes).copy()

    # Garage information
    X["garage"] = homes["garage"].fillna(-1)

    # Lot area: frontage x depth
    X["lot_area"] = (
        homes["lot_front"] * homes["lot_depth"]
    ).fillna(-1)

    # Whether lot measurements are available
    X["has_lot_info"] = (
        homes["lot_front"].notna().astype(int)
    )

    # Convert categorical variables into numerical columns
    categorical = pd.get_dummies(
        homes[["municipality", "community", "house_style"]],
        dtype=int
    )

    X = pd.concat([X, categorical], axis=1)

    return X

In [13]:
X_train_improved = improved_features(
    data.hide_answers(train, "off_market")
)

X_valid_improved = improved_features(
    data.hide_answers(valid, "off_market")
)

# Align validation features with training features
X_valid_improved = X_valid_improved.reindex(
    columns=X_train_improved.columns,
    fill_value=0
)

print("Original features:", X_train.shape[1])
print("Improved features:", X_train_improved.shape[1])

print("Training:", X_train_improved.shape)
print("Validation:", X_valid_improved.shape)

print("Missing values:", X_train_improved.isna().sum().sum())

display(X_train_improved.head())

Original features: 13
Improved features: 182
Training: (147709, 182)
Validation: (2291, 182)
Missing values: 0


,square_feet,bedrooms,washrooms,parking,latitude,longitude,months,is_Condo Apt,is_Detached,is_Semi-Detached,...,house_style_Backsplit 3,house_style_Backsplit 4,house_style_Backsplit 5,house_style_Bungalow,house_style_Bungalow-Raised,house_style_Loft,house_style_Multi-Level,house_style_Sidesplit 3,house_style_Sidesplit 4,house_style_Stacked Townhse
0,622.0,1,1,0,43.602039,-79.261184,-0.098555,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,649.0,2,1,0,43.695388,-79.510865,-0.197109,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,999.5,2,1,1,43.740512,-79.386216,-0.328515,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,876.0,1,1,1,43.643774,-79.470180,-0.427070,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,514.0,1,1,1,43.696172,-79.274137,-0.886991,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
improved_results_list = []

for month in validation_months:
    start = month.start_time
    end = (month + 1).start_time

    train_month = listings[
        listings["date_sold"] < start
    ].copy()

    valid_month = listings[
        (listings["date_sold"] >= start) &
        (listings["date_sold"] < end)
    ].copy()

    # Hide answers to prevent data leakage
    train_homes = data.hide_answers(
        train_month, "off_market"
    )

    valid_homes = data.hide_answers(
        valid_month, "off_market"
    )

    # Create improved features
    X_train_new = improved_features(train_homes)
    X_valid_new = improved_features(valid_homes)

    X_valid_new = X_valid_new.reindex(
        columns=X_train_new.columns,
        fill_value=0
    )

    # Training target
    y_train_new = np.log(train_month["sold_price"])

    # Same model as Experiment 1
    improved_model = HistGradientBoostingRegressor(
        random_state=42
    )

    improved_model.fit(X_train_new, y_train_new)

    # Predict selling prices
    predictions = np.exp(
        improved_model.predict(X_valid_new)
    )

    # Calculate error
    monthly_results = pd.DataFrame({
        "month": str(month),
        "actual": valid_month["sold_price"].to_numpy(),
        "predicted": predictions
    })

    monthly_results["APE"] = (
        abs(monthly_results["predicted"] - monthly_results["actual"])
        / monthly_results["actual"]
    )

    improved_results_list.append(monthly_results)

    print(
        f"{month}: "
        f"MdAPE = {monthly_results['APE'].median() * 100:.2f}%"
    )

improved_results = pd.concat(
    improved_results_list,
    ignore_index=True
)

improved_mdape = improved_results["APE"].median() * 100

print(f"\nImproved Model MdAPE: {improved_mdape:.2f}%")

2025-12: MdAPE = 12.40%
2026-01: MdAPE = 11.95%
2026-02: MdAPE = 11.88%
2026-03: MdAPE = 11.35%
2026-04: MdAPE = 11.46%
2026-05: MdAPE = 11.67%

Improved Model MdAPE: 11.72%


In [15]:
comparison = pd.DataFrame({
    "Model": [
        "Starter Baseline",
        "Original Gradient Boosting",
        "Improved Gradient Boosting"
    ],
    "MdAPE (%)": [
        20.84,
        14.87,
        improved_mdape
    ]
})

display(comparison)

improvement = 14.87 - improved_mdape

print(f"Change: {improvement:+.2f} percentage points")

,Model,MdAPE (%)
0,Starter Baseline,20.840000
1,Original Gradient Boosting,14.870000
2,Improved Gradient Boosting,11.722832


Change: +3.15 percentage points


### Experiment 2 — Results

Additional property and location features were added to the original 13-feature Gradient Boosting model, resulting in 182 features.

The new features included garage information, lot area, an indicator for lot information availability, municipality, community, and house style.

The model architecture and hyperparameters were kept unchanged to isolate the overall effect of feature engineering.

**Results:**
- Original model MdAPE: 14.87%
- Improved model MdAPE: 11.72%
- Improvement: 3.15 percentage points (approximately 21.2% relative reduction)

The additional features substantially improved prediction accuracy across the six-month walk-forward validation period.

However, since multiple features were introduced simultaneously, further experiments are needed to determine which individual feature groups contributed most to the improvement.

**Next step:** Conduct feature ablation experiments and explore alternative models or hyperparameter settings.

In [16]:
def no_community_features(homes):
    X = improved_features(homes)

    community_columns = [
        col for col in X.columns
        if col.startswith("community_")
    ]

    return X.drop(columns=community_columns)

In [18]:
X_train_new = no_community_features(train_homes)
X_valid_new = no_community_features(valid_homes)

In [19]:
no_community_results = pd.concat(
    improved_results_list,
    ignore_index=True
)

no_community_mdape = (
    no_community_results["APE"].median() * 100
)

print(f"No Community MdAPE: {no_community_mdape:.2f}%")

No Community MdAPE: 11.72%


In [20]:
comparison = pd.DataFrame({
    "Model": [
        "Original Gradient Boosting",
        "All Improved Features",
        "Without Community"
    ],
    "MdAPE (%)": [
        14.87,
        improved_mdape,
        no_community_mdape
    ]
})

display(comparison)

difference = no_community_mdape - improved_mdape

print(f"Difference: {difference:+.2f} percentage points")

,Model,MdAPE (%)
0,Original Gradient Boosting,14.870000
1,All Improved Features,11.722832
2,Without Community,11.722832


Difference: +0.00 percentage points


In [21]:
X_test = no_community_features(
    data.hide_answers(train, "off_market")
)

community_cols = [
    col for col in X_test.columns
    if col.startswith("community_")
]

print("Remaining community columns:", len(community_cols))
print("Total features:", X_test.shape[1])

Remaining community columns: 0
Total features: 37


In [22]:
ablation_results = []

for month in validation_months:
    start = month.start_time
    end = (month + 1).start_time

    train_month = listings[
        listings["date_sold"] < start
    ].copy()

    valid_month = listings[
        (listings["date_sold"] >= start) &
        (listings["date_sold"] < end)
    ].copy()

    train_homes = data.hide_answers(
        train_month, "off_market"
    )

    valid_homes = data.hide_answers(
        valid_month, "off_market"
    )

    # Remove community features
    X_train_ablation = no_community_features(train_homes)
    X_valid_ablation = no_community_features(valid_homes)

    X_valid_ablation = X_valid_ablation.reindex(
        columns=X_train_ablation.columns,
        fill_value=0
    )

    print(
        f"{month}: Training with "
        f"{X_train_ablation.shape[1]} features"
    )

    # Train a NEW model
    ablation_model = HistGradientBoostingRegressor(
        random_state=42
    )

    ablation_model.fit(
        X_train_ablation,
        np.log(train_month["sold_price"])
    )

    predictions = np.exp(
        ablation_model.predict(X_valid_ablation)
    )

    monthly_results = pd.DataFrame({
        "month": str(month),
        "actual": valid_month["sold_price"].to_numpy(),
        "predicted": predictions
    })

    monthly_results["APE"] = (
        abs(monthly_results["predicted"] - monthly_results["actual"])
        / monthly_results["actual"]
    )

    ablation_results.append(monthly_results)

    print(
        f"MdAPE = "
        f"{monthly_results['APE'].median() * 100:.2f}%"
    )

ablation_df = pd.concat(ablation_results, ignore_index=True)

no_community_mdape = ablation_df["APE"].median() * 100

print(f"\nWITHOUT COMMUNITY: {no_community_mdape:.4f}%")
print(f"WITH COMMUNITY: {improved_mdape:.4f}%")
print(
    f"Difference: "
    f"{no_community_mdape - improved_mdape:+.4f} pp"
)

2025-12: Training with 37 features
MdAPE = 15.30%
2026-01: Training with 37 features
MdAPE = 15.10%
2026-02: Training with 37 features
MdAPE = 13.10%
2026-03: Training with 37 features
MdAPE = 13.17%
2026-04: Training with 37 features
MdAPE = 13.36%
2026-05: Training with 37 features
MdAPE = 13.80%

WITHOUT COMMUNITY: 13.7118%
WITH COMMUNITY: 11.7228%
Difference: +1.9889 pp
